In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## Use case Intro:

### EMG signal classification
 > Source: https://archive.ics.uci.edu/ml/datasets/EMG+Physical+Action+Data+Set

#### Imagine You are a Decision Scientist at Boston Dynamics( a robotics company)

- Your team is making a **robotics arm that can be controlled by brain signals**.
- These brain signals are recorded through **EMG**.

#### Problem Statement:
- Your task is to classify these EMG signals into 20 different physical actions
- This will then be used for controlling the robotics arm.

#### What is EMG (ElectroMioGraphy) ?
  - Technique to study electrical signals produced by muscular movement.

#### Dataset
- You have a dataset of EMG signals from 4 subjects/people.

#### How was the data collected ?
  - Subject was asked to perform specific physical actions
  - Signals produced due to that movement were recorded over time.
  - 8 channels were used to record the signals
  - Channels here correspond to muscles\
    For eg: Right-hand bicep
  - Frequency : 10 $ms^{-1}$

Now, lets import some libs at first.

In [2]:
%%time
!gdown 1h86M8si2YT-aI4Zec1MeMP_mPYsLPy5F

CPU times: total: 109 ms
Wall time: 9.54 s


Downloading...
From: https://drive.google.com/uc?id=1h86M8si2YT-aI4Zec1MeMP_mPYsLPy5F
To: C:\Users\dipty\ipynb\Courses\Scaler_ML_self\3ML_supervised\11SCALER\M1\W3\10_6_24\emg.rar

  0%|          | 0.00/18.6M [00:00<?, ?B/s]
  3%|2         | 524k/18.6M [00:00<00:08, 2.05MB/s]
  8%|8         | 1.57M/18.6M [00:00<00:03, 4.76MB/s]
 28%|##8       | 5.24M/18.6M [00:00<00:00, 14.2MB/s]
 39%|###9      | 7.34M/18.6M [00:00<00:00, 14.8MB/s]
 51%|#####     | 9.44M/18.6M [00:00<00:00, 9.77MB/s]
 62%|######2   | 11.5M/18.6M [00:01<00:00, 11.7MB/s]
 70%|#######   | 13.1M/18.6M [00:01<00:00, 12.0MB/s]
 79%|#######8  | 14.7M/18.6M [00:01<00:00, 12.1MB/s]
 87%|########7 | 16.3M/18.6M [00:01<00:00, 12.5MB/s]
 96%|#########5| 17.8M/18.6M [00:01<00:00, 12.4MB/s]
100%|##########| 18.6M/18.6M [00:01<00:00, 11.4MB/s]


In [3]:
# x is extract

!unrar x "./emg.rar" "./"

'unrar' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
# list files
!ls ./EMG\ Physical\ Action\ Data\ Set/sub1/Aggressive/txt/

'ls' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
!cat ./EMG\ Physical\ Action\ Data\ Set/sub1/Aggressive/txt/Slapping.txt

'cat' is not recognized as an internal or external command,
operable program or batch file.


In [6]:
actions = {}
data_dirs = ["./EMG Physical Action Data Set/sub1/Aggressive/txt",
             "./EMG Physical Action Data Set/sub1/Normal/txt"]

ind = 0
data = pd.DataFrame()

for dirs in data_dirs:
    for files in os.listdir(dirs):
        with open(os.path.join(dirs, files), "r") as f:
            temp = pd.read_csv(f.name,
                               sep="\t",
                               header=None,
                               names=["ch" + str(i) for i in range(1, 9)])  # 8 input channels

            # chunking using Max of every 10 sequential values.
            temp_chunked = pd.DataFrame()
            for i in range(0, len(temp), 10):
                temp_chunked = pd.concat([temp_chunked, temp.iloc[i:i + 10].max().to_frame().T], ignore_index=True)

            labels = [files[:-4] for _ in range(len(temp_chunked))]  # remove the last 4 characters=".txt" from the filename
            actions[files[:-4]] = ind
            temp_chunked["Action"] = labels

            data = pd.concat([data, temp_chunked])

        ind += 1

print(actions)

{'Elbowing': 0, 'Frontkicking': 1, 'Hamering': 2, 'Headering': 3, 'Kneeing': 4, 'Pulling': 5, 'Punching': 6, 'Pushing': 7, 'Sidekicking': 8, 'Slapping': 9, 'Bowing': 10, 'Clapping': 11, 'Handshaking': 12, 'Hugging': 13, 'Jumping': 14, 'Running': 15, 'Seating': 16, 'Standing': 17, 'Walking': 18, 'Waving': 19}


In [7]:
data.head()

,ch1,ch2,ch3,ch4,ch5,ch6,ch7,ch8,Action
0,717,391,2615,-29,4000,205,1084,4000,Elbowing
1,1036,251,2989,162,4000,2971,3062,4000,Elbowing
2,3705,30,4000,549,4000,2940,-1767,-205,Elbowing
3,2679,347,1566,167,-4000,2758,-3965,785,Elbowing
4,1689,77,4000,-246,4000,2422,-1767,360,Elbowing


In [8]:
data.shape

(19711, 9)

In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19711 entries, 0 to 999
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ch1     19711 non-null  int64 
 1   ch2     19711 non-null  int64 
 2   ch3     19711 non-null  int64 
 3   ch4     19711 non-null  int64 
 4   ch5     19711 non-null  int64 
 5   ch6     19711 non-null  int64 
 6   ch7     19711 non-null  int64 
 7   ch8     19711 non-null  int64 
 8   Action  19711 non-null  object
dtypes: int64(8), object(1)
memory usage: 1.5+ MB


In [10]:
Y = data["Action"]
X = data.drop(columns = ["Action"])

In [11]:
print(Y.unique())

['Elbowing' 'Frontkicking' 'Hamering' 'Headering' 'Kneeing' 'Pulling'
 'Punching' 'Pushing' 'Sidekicking' 'Slapping' 'Bowing' 'Clapping'
 'Handshaking' 'Hugging' 'Jumping' 'Running' 'Seating' 'Standing'
 'Walking' 'Waving']


In [12]:
Y = Y.map(actions)
Y.head()

0    0
1    0
2    0
3    0
4    0
Name: Action, dtype: int64

In [13]:
X = abs(X)
X.head()

,ch1,ch2,ch3,ch4,ch5,ch6,ch7,ch8
0,717,391,2615,29,4000,205,1084,4000
1,1036,251,2989,162,4000,2971,3062,4000
2,3705,30,4000,549,4000,2940,1767,205
3,2679,347,1566,167,4000,2758,3965,785
4,1689,77,4000,246,4000,2422,1767,360


In [14]:
# moving average.

X = X.ewm(10).mean()

In [15]:
from sklearn.model_selection import train_test_split

X = np.array(X.values)
Y = np.array(Y.values)

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, shuffle = True, random_state=3)

print(f"Sizes of the sets created are:\nTraining set:{X_train.shape[0]}\nTest set:{X_test.shape[0]}")

Sizes of the sets created are:
Training set:15768
Test set:3943


# Decision Tree

https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

In [16]:
%%time
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate

tree_clf = DecisionTreeClassifier(random_state=7, max_depth = 7, )
cv_acc_results = cross_validate(tree_clf, X_train, Y_train, cv = 5, scoring = 'accuracy', return_train_score = True,n_jobs=-1)

print(f"K-Fold Accuracy Mean: Train: {cv_acc_results['train_score'].mean().round(3)*100} Validation: {cv_acc_results['test_score'].mean().round(3)*100}")

K-Fold Accuracy Mean: Train: 68.7 Validation: 67.9
CPU times: total: 797 ms
Wall time: 1.11 s


# RandomForest

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [17]:
%%time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

tree_clf = RandomForestClassifier(random_state=7, max_depth = 7, n_estimators= 100 )
cv_acc_results = cross_validate(tree_clf, X_train, Y_train, cv = 5, scoring = 'accuracy', return_train_score = True,n_jobs=-1)

print(f"K-Fold Accuracy Mean: Train: {cv_acc_results['train_score'].mean().round(3)*100} Validation: {cv_acc_results['test_score'].mean().round(3)*100}")


K-Fold Accuracy Mean: Train: 84.39999999999999 Validation: 83.1
CPU times: total: 172 ms
Wall time: 4.38 s


# GBDT

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html

In [18]:
%%time
from sklearn.ensemble import GradientBoostingClassifier


tree_clf = GradientBoostingClassifier(random_state=7, max_depth = 4, n_estimators= 150, learning_rate = 0.1 )
cv_acc_results = cross_validate(tree_clf, X_train, Y_train, cv = 3, scoring = 'accuracy', return_train_score = True,n_jobs=-1)

print(f"K-Fold Accuracy Mean: Train: {cv_acc_results['train_score'].mean().round(3)*100} Validation: {cv_acc_results['test_score'].mean().round(3)*100}")


K-Fold Accuracy Mean: Train: 100.0 Validation: 94.39999999999999
CPU times: total: 15.6 ms
Wall time: 7min 12s


# GBDT >> AdaBoost

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html


> For loss ‘exponential’, gradient boosting recovers the AdaBoost algorithm.

In [19]:
%%time
from sklearn.ensemble import AdaBoostClassifier


tree_clf = AdaBoostClassifier(random_state=7, n_estimators= 150, learning_rate = 0.1, base_estimator=  DecisionTreeClassifier(random_state=7, max_depth = 4 ))
cv_acc_results = cross_validate(tree_clf, X_train, Y_train, cv = 3, scoring = 'accuracy', return_train_score = True,n_jobs=-1)

print(f"K-Fold Accuracy Mean: Train: {cv_acc_results['train_score'].mean().round(3)*100} Validation: {cv_acc_results['test_score'].mean().round(3)*100}")


K-Fold Accuracy Mean: Train: 80.80000000000001 Validation: 78.8
CPU times: total: 37.9 s
Wall time: 1min 14s


# GBDT >> Stochastic Gradient Boosting

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html


**subsample** [tackles high variance like Random Forest]

float, default=1.0

The fraction of samples to be used for fitting the individual base learners. If smaller than 1.0 this results in Stochastic Gradient Boosting. `subsample` interacts with the parameter `n_estimators`. Choosing `subsample < 1.0` leads to a reduction of variance and an increase in bias. Values must be in the range `(0.0, 1.0]`.

In [20]:
%%time
from sklearn.ensemble import GradientBoostingClassifier


tree_clf = GradientBoostingClassifier(random_state=7, max_depth = 4, n_estimators= 150, learning_rate = 0.1, subsample=0.75 )
cv_acc_results = cross_validate(tree_clf, X_train, Y_train, cv = 3, scoring = 'accuracy', return_train_score = True,n_jobs=-1)

print(f"K-Fold Accuracy Mean: Train: {cv_acc_results['train_score'].mean().round(3)*100} Validation: {cv_acc_results['test_score'].mean().round(3)*100}")


K-Fold Accuracy Mean: Train: 100.0 Validation: 94.6
CPU times: total: 6min
Wall time: 14min 57s


# XGboost

XGBoost Docs: https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBClassifier

XGBoost (Research Paper): A Scalable Tree Boosting System: https://www.kdd.org/kdd2016/papers/files/rfp0697-chenAemb.pdf

What it does:

1. parallelised feature selection
1. parallelised on seperators
1. Row and column sampling
1. histogram based binning in numerical feature

Other facts:

1. Research project of 2016 that is adopted in many other programming languages
1. Can use

Hyper parameters:

- gamma (Optional[float])

    – (min_split_loss) Minimum loss reduction required to make a further partition on a leaf node of the tree.

- colsample_bytree (Optional[float])
    
    – Subsample ratio of columns when constructing each tree.

- colsample_bylevel (Optional[float])
    
    – Subsample ratio of columns for each level.

- colsample_bynode (Optional[float])
    
    – Subsample ratio of columns for each split.

In [25]:
%%time
# library
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=100, objective='multi:softmax', num_class=20, subsample=0.8, max_depth=4, learning_rate=0.2, colsample_bytree=0.8, colsample_bylevel=1, silent=True)
xgb.fit(X_train, Y_train)


print( xgb.score(X_train, Y_train))
print( xgb.score(X_test, Y_test))

0.9939117199391172
0.9637331980725337
CPU times: total: 32.3 s
Wall time: 10.5 s


# LightGBM
by Microsoft(2017)

https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMClassifier.html#lightgbm-lgbmclassifier

1. Faster than XG Boost
1. GOSS (Gradient Based One Side Sampling): drop small residuals/error [Intelligent Row Sampling]
1. Exclusive Feature Bundling (using graph based approaches)

Read more:
https://towardsdatascience.com/what-makes-lightgbm-lightning-fast-a27cf0d9785e

In [27]:
%%time
from lightgbm import LGBMClassifier
lgm = LGBMClassifier()
lgm.fit(X_train, Y_train)


print( lgm.score(X_train, Y_train))
print( lgm.score(X_test, Y_test))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000794 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 15768, number of used features: 8
[LightGBM] [Info] Start training from score -3.000054
[LightGBM] [Info] Start training from score -2.998781
[LightGBM] [Info] Start training from score -3.009011
[LightGBM] [Info] Start training from score -2.982377
[LightGBM] [Info] Start training from score -2.956434
[LightGBM] [Info] Start training from score -3.023251
[LightGBM] [Info] Start training from score -3.032419
[LightGBM] [Info] Start training from score -3.002605
[LightGBM] [Info] Start training from score -2.986139
[LightGBM] [Info] Start training from score -3.011585
[LightGBM] [Info] Start training from score -3.018049
[LightGBM] [Info] Start training from score -2.993705
[LightGBM] [Info] Start training from score -3.010297
[LightGBM]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

# StackingClassifier

https://rasbt.github.io/mlxtend/user_guide/classifier/StackingClassifier/

![](https://rasbt.github.io/mlxtend/user_guide/classifier/StackingClassifier_files/stackingclassification_overview.png)

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html



In [23]:
# %%time
# meta models/ not used in production
# too expensive
# clf1 = Logreg()
# clf2 = DT()
# clf3 = GBDT()
# meta_clf = DT()


# model = StackingClassifier( [clf1, clf2, clf3] , meta_clf)
# model.fit()


# model.predict()
# model.score()

# Cascading

<center><img src='https://drive.google.com/uc?id=1fF1yaPhgX5ytbAHrwon5WltkV8_VlDGG' width=800></center>

- cascading is used in industry where
    -  loss associated with misclassification is high

For example:
- cancer detection
- financial domain etc